## Setup

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import re
import xarray as xr
import glob
from tqdm import tqdm
import dateutil.parser
import astropy.units as u
from astropy.coordinates import SkyCoord
import sunpy.coordinates

In [ ]:
load_dotenv()
RAW_PATH = os.getenv('EVENTS_RAW_PATH')
PROCESSED_PATH = os.getenv('EVENTS_PROCESSED_PATH')

DATA_YEARS = range(2010, 2025)

## Read Files and create Intermediate CSVs

In [ ]:
def parse_ftp_txt(filepath):
    """Lê arquivos .txt do SWPC, extraindo a data do cabeçalho para cruzar com os horários."""
    data = []
    current_date = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith(':Date:'):
                current_date = line.replace(':Date:', '').strip().replace(' ', '-')
            elif line and not line.startswith('#') and not line.startswith(':'):
                parts = line.split()
                if current_date and len(parts) >= 9:
                    data.append([current_date] + parts)

    cols = ['Date', 'Event', 'Begin', 'Max', 'End', 'Obs', 'Q', 'Type', 'Loc_Frq', 'Particulars', 'Reg']
    df_ = pd.DataFrame(data)
    if not df_.empty and df_.shape[1] >= len(cols):
        df_ = df_.iloc[:, :len(cols)]
        df_.columns = cols
    return df_

def parse_srs_txt(filepath):
    """Extrai a tabela principal 'I. Regions with Sunspots' e a data de emissão."""
    data = []
    current_date = None

    with open(filepath, 'r') as f:
        lines = f.readlines()

    in_table = False
    for line in lines:
        if line.startswith(':Issued:'):
            date_str = line.replace(':Issued:', '').replace('UTC', '').strip()
            try:
                current_date = pd.to_datetime(date_str).strftime('%Y-%m-%d')
            except:
                pass

        if 'I. Regions with Sunspots' in line:
            in_table = True
            continue
        if in_table and (line.startswith('IA.') or line.startswith('II.')):
            break
        if in_table:
            parts = line.strip().split()
            if len(parts) >= 8 and parts[0].isdigit():
                data.append([current_date] + parts)

    cols = ['Date', 'Nmbr', 'Location', 'Lo', 'Area', 'Z', 'LL', 'NN', 'Mag_Type']
    df_ = pd.DataFrame(data, columns=cols)
    return df_

In [ ]:
for y_ in tqdm(DATA_YEARS, desc="Processando Anos"):
    y_str = str(y_)
    output_dir = os.path.join(PROCESSED_PATH, y_str)
    os.makedirs(output_dir, exist_ok=True)

    # 1. Events
    events_files = glob.glob(os.path.join(RAW_PATH, y_str, '*', 'sci_xrsf-l2-flsum_*.nc'))
    df_events_list = []
    for f in events_files:
        try:
            ds = xr.open_dataset(f)
            df = ds.to_dataframe().reset_index()
            df_events_list.append(df)
            ds.close()
        except Exception as e:
            print(f"Erro ao ler NC (Events): {f} - {e}")
    if df_events_list:
        pd.concat(df_events_list, ignore_index=True).to_csv(os.path.join(output_dir, f'events_{y_str}.csv'), index=False)

    # 2. FTP
    swpc_files = glob.glob(os.path.join(RAW_PATH, y_str, f'{y_str}_SWPC_events', f'{y_str}_events', '*events.txt'))
    df_swpc_list = [parse_ftp_txt(f) for f in swpc_files]
    df_swpc_list = [df for df in df_swpc_list if not df.empty]
    if df_swpc_list:
        pd.concat(df_swpc_list, ignore_index=True).to_csv(os.path.join(output_dir, f'FTP_{y_str}.csv'), index=False)

    # 3. SSW
    ssw_files = glob.glob(os.path.join(RAW_PATH, y_str, 'SSW', 'SSW_*.csv'))
    df_ssw_list = [pd.read_csv(f) for f in ssw_files]
    if df_ssw_list:
        pd.concat(df_ssw_list, ignore_index=True).to_csv(os.path.join(output_dir, f'SSW_{y_str}.csv'), index=False)

    # 4. Locations
    if y_ >= 2017:
        locations_files = glob.glob(os.path.join(RAW_PATH, y_str, 'NCEI_FLLOC', '*', 'sci_xrsf-l2-flloc_*.nc'))
        df_loc_list = []
        for f in locations_files:
            try:
                ds = xr.open_dataset(f)
                df = ds.to_dataframe().reset_index()
                df_loc_list.append(df)
                ds.close()
            except Exception as e:
                print(f"Erro ao ler NC (Locations): {f} - {e}")
        if df_loc_list:
            pd.concat(df_loc_list, ignore_index=True).to_csv(os.path.join(output_dir, f'Locations_{y_str}.csv'), index=False)

    # 5. Reports
    srs_files = glob.glob(os.path.join(RAW_PATH, y_str, f'{y_str}_SWPC_SRS', f'{y_str}_SRS', '*'))
    df_srs_list = [parse_srs_txt(f) for f in srs_files]
    df_srs_list = [df for df in df_srs_list if not df.empty]
    if df_srs_list:
        pd.concat(df_srs_list, ignore_index=True).to_csv(os.path.join(output_dir, f'SRS_{y_str}.csv'), index=False)

## Read Intermediate CSVs

In [ ]:
PROCESSED_DATA = {}
DATASETS = ['events', 'FTP', 'SSW', 'Locations', 'SRS']

for year in tqdm(range(2010, 2025), desc="Carregando CSVs Processados"):
    y_str = str(year)
    PROCESSED_DATA[y_str] = {}

    for dataset in DATASETS:
        if dataset == 'Locations' and year < 2017:
            PROCESSED_DATA[y_str][dataset] = pd.DataFrame()
            continue

        file_path = os.path.join(PROCESSED_PATH, y_str, f'{dataset}_{y_str}.csv')

        if os.path.exists(file_path):
            try:
                PROCESSED_DATA[y_str][dataset] = pd.read_csv(file_path)
            except pd.errors.EmptyDataError:
                PROCESSED_DATA[y_str][dataset] = pd.DataFrame()
        else:
            PROCESSED_DATA[y_str][dataset] = pd.DataFrame()

## Preparação, Padronização e Correção das Bases

In [ ]:
# =============================================================================
# 1.1 UNIFICAÇÃO TEMPORAL DOS DADOS (Carga na Memória)
# =============================================================================
# Os dados brutos foram previamente extraídos, processados e armazenados em fatias anuais (dicionário PROCESSED_DATA) para otimização de leitura.
# No entanto, o algoritmo de matching cruzado e a análise de séries temporais exigem uma visão contínua do tempo. Concatenamos as fatias anuais em DataFrames globais e únicos para permitir operações vetorizadas e garantir que eventos que ocorrem nas viradas de ano sejam comparados e ordenados corretamente.
df_sci_list, df_ftp_list, df_ssw_list, df_srs_list = [], [], [], []

for y in DATA_YEARS:
    y_str = str(y)
    if not PROCESSED_DATA[y_str]['events'].empty:
        df_sci_list.append(PROCESSED_DATA[y_str]['events'])
    if not PROCESSED_DATA[y_str]['FTP'].empty:
        df_ftp_list.append(PROCESSED_DATA[y_str]['FTP'])
    if not PROCESSED_DATA[y_str]['SSW'].empty:
        df_ssw_list.append(PROCESSED_DATA[y_str]['SSW'])
    if not PROCESSED_DATA[y_str]['SRS'].empty:
        df_srs_list.append(PROCESSED_DATA[y_str]['SRS'])

df_sci_raw = pd.concat(df_sci_list, ignore_index=True) if df_sci_list else pd.DataFrame()
df_ftp_raw = pd.concat(df_ftp_list, ignore_index=True) if df_ftp_list else pd.DataFrame()
df_ssw_raw = pd.concat(df_ssw_list, ignore_index=True) if df_ssw_list else pd.DataFrame()

df_srs_raw = pd.concat(df_srs_list, ignore_index=True) if df_srs_list else pd.DataFrame()
if not df_srs_raw.empty:
    df_srs_raw['Date'] = pd.to_datetime(df_srs_raw['Date'], errors='coerce').dt.date

In [ ]:
# =============================================================================
# 1.2 PADRONIZAÇÃO DE ESCALAS, TEMPO E METADADOS
# =============================================================================
# Criar uma base comum para viabilizar a junção (matching) de catálogos heterogêneos.
# 1. Escala de Intensidade: Catálogos operacionais (FTP/SSW) usam classificações alfanuméricas (ex: C2.2, M1.5)[cite: 2]. A base Science-Quality reporta fluxo em W/m^2. A função `goes_class_to_log10` converte a letra (A, B, C, M, X) e o multiplicador para a escala logarítmica contínua (log10), permitindo aplicar a tolerânciamatemática de <= 0.3 estabelecida pelo artigo de referência (defects_and_inconsistencies).
# 2. Alinhamento Temporal: Como o matching depende da proximidade temporal entre os picos das explosões, isolamos estritamente os eventos 'EVENT_PEAK' na base Science-Quality e reconstruímos strings de data/hora fragmentadas nas bases operacionais para o formato datetime (UTC) universal.
# 3. Resgate de Regiões Ativas (AR): Essencial para conectar o 'flare' às métricas magnéticas regionais (SHARPs). Na base SSW, o número da AR não tem coluna própria, ficando concatenado como ruído em 'Derived Position' (ex: "S08W89 ( 3917 )").
def goes_class_to_log10(flare_class):
    if pd.isna(flare_class) or not isinstance(flare_class, str) or len(flare_class) < 2:
        return np.nan
    scale = {'A': -8, 'B': -7, 'C': -6, 'M': -5, 'X': -4}
    letter = flare_class[0].upper()
    if letter not in scale:
        return np.nan
    try:
        multiplier = float(flare_class[1:])
        return np.log10(multiplier) + scale[letter]
    except ValueError:
        return np.nan

# --- EVENTS ---
# Filtramos por 'EVENT_PEAK'.
df_sci = df_sci_raw[df_sci_raw['status'] == 'EVENT_PEAK'].copy()
df_sci['peak_time'] = pd.to_datetime(df_sci['time'], utc=True)
df_sci['log10_intensity'] = np.log10(df_sci['xrsb_flux'])
df_sci = df_sci[['flare_id', 'peak_time', 'flare_class', 'log10_intensity']].copy()

# --- SSW ---
df_ssw = df_ssw_raw.copy()
# Extrair data de 'Start' e fundir com a hora em 'Peak'
df_ssw['date_str'] = df_ssw['Start'].str[:10]
df_ssw['peak_time'] = pd.to_datetime(df_ssw['date_str'] + ' ' + df_ssw['Peak'], utc=True)
df_ssw['log10_intensity'] = df_ssw['GOES Class'].apply(goes_class_to_log10)
df_ssw['Reg'] = df_ssw['Derived Position'].str.extract(r'\(\s*(\d+)\s*\)')

df_ssw = df_ssw[['EName', 'peak_time', 'GOES Class', 'log10_intensity', 'Reg']].dropna(subset=['peak_time'])

# --- FTP ---
df_ftp = df_ftp_raw.copy()
if 'Date' in df_ftp.columns and 'Max' in df_ftp.columns:
    df_ftp['Max_time'] = df_ftp['Max'].astype(str).str.zfill(4).apply(lambda x: f"{x[:2]}:{x[2:]}:00" if x.isdigit() else np.nan)
    df_ftp['peak_time'] = pd.to_datetime(df_ftp['Date'] + ' ' + df_ftp['Max_time'], errors='coerce', utc=True)
    df_ftp['log10_intensity'] = df_ftp['Particulars'].apply(goes_class_to_log10)
    df_ftp = df_ftp[['Event', 'peak_time', 'Particulars', 'log10_intensity', 'Reg']].dropna(subset=['peak_time'])

In [ ]:
# =============================================================================
# 1.3 CORREÇÃO DE ESCALA SWPC E SSW
# =============================================================================
# Os dados da base Science-Quality (arquivos .nc) já foram reprocessados pela NCEI e não possuem o fator de escala artificial.
# No entanto, antes de 2019-12-09 (quando o GOES-16 se tornou o primário), a SWPC aplicava um fator de escala (0.7) nos dados operacionais, puxando as intensidades para baixo. A base SSW herdou esse mesmo viés.
# Para garantir que o cruzamento (Matching) no próximo passo compare "maçãs com maçãs", precisamos elevar as bases operacionais (FTP e SSW) para a escala física real da base Science-Quality, removendo o fator de 0.7.
# Matematicamente: log10(Fluxo / 0.7) = log10(Fluxo) - log10(0.7)
cutoff_date = pd.to_datetime('2019-12-09', utc=True)

mask_ssw = df_ssw['peak_time'] < cutoff_date
df_ssw.loc[mask_ssw, 'log10_intensity'] += 0.15

if not df_ftp.empty:
    mask_ftp = df_ftp['peak_time'] < cutoff_date
    df_ftp.loc[mask_ftp, 'log10_intensity'] += 0.15

print(f"✅ Tratamento Inicial Concluído:")
print(f"Science-Quality: {len(df_sci)} eventos de pico prontos.")
print(f"SSW: {len(df_ssw)} eventos padronizados e escala corrigida.")
print(f"FTP: {len(df_ftp)} eventos padronizados e escala corrigida.")

## Algoritmo de Matching (Science-Quality <-> Operacionais)

In [ ]:
# Para cada flare na base Science-Quality, buscamos a correspondência nos catálogos operacionais (FTP e SSW) para herdar o número da Região Ativa (AR).
# Critérios estabelecidos na literatura (Hu et al., 2025):
# 1. Delta_Tempo_Pico <= 15 minutos
# 2. Delta_Intensidade_Log10 <= 0.3
# 3. Desempate: Menor Delta_Tempo_Pico
# Estratégia em Cascata: Como o SWPC-FTP é o catálogo oficial primário, buscamos nele primeiro. Se não houver 'match', buscamos no catálogo SSW.

def find_match(target_row, df_catalogue, time_tol_min=15, flux_tol_log=0.3):
    """Encontra o melhor match num catálogo dado as tolerâncias."""
    if df_catalogue.empty or pd.isna(target_row['peak_time']) or pd.isna(target_row['log10_intensity']):
        return None

    time_diffs = (df_catalogue['peak_time'] - target_row['peak_time']).abs().dt.total_seconds() / 60.0
    flux_diffs = (df_catalogue['log10_intensity'] - target_row['log10_intensity']).abs()

    mask = (time_diffs <= time_tol_min) & (flux_diffs <= flux_tol_log)
    candidates = df_catalogue[mask].copy()

    if candidates.empty:
        return None

    candidates['time_diff'] = time_diffs[mask]
    best_match = candidates.sort_values(by='time_diff').iloc[0]

    return best_match['Reg']

In [ ]:
df_sci['matched_AR'] = np.nan
df_sci['match_source'] = np.nan

print("Iniciando Matching (Isso pode levar alguns segundos)...")

for idx, row in tqdm(df_sci.iterrows(), total=len(df_sci), desc="Cruzando Catálogos"):

    # 1.
    ar_match = find_match(row, df_ftp)
    source = 'FTP'

    # 2.
    if pd.isna(ar_match):
        ar_match = find_match(row, df_ssw)
        source = 'SSW' if pd.notna(ar_match) else np.nan

    # 3.
    if pd.notna(ar_match):
        df_sci.at[idx, 'matched_AR'] = ar_match
        df_sci.at[idx, 'match_source'] = source

total_sci = len(df_sci)
matched_count = df_sci['matched_AR'].notna().sum()
success_rate = (matched_count / total_sci) * 100

print(f"\n✅ Matching Concluído:")
print(f"Total de flares Science-Quality: {total_sci}")
print(f"Flares pareados com sucesso: {matched_count} ({success_rate:.1f}%)")
print(f"Flares sem AR atribuída (sem match): {total_sci - matched_count}")

## Preparação e Pivotamento Para o Augmentation Geométrico

In [ ]:
# Os eventos que não conseguiram correspondência temporal/intensidade nos catálogos operacionais (ou que receberam AR 0) precisarão ter sua Região Ativa inferida por proximidade espacial. Isso só é possível para dados a partir de 2017-02-09, quando a série GOES-R passou a fornecer medições de localização precisas.
# Obstáculo de Formato: O arquivo NCEI Locations guarda as coordenadas HPC (Helioprojective Cartesian) em formato longo:
# Linha A -> coordinate = 0 (Eixo X)
# Linha B -> coordinate = 1 (Eixo Y)
# Usaremos pd.pivot_table para transformar isso em formato largo (colunas hpc_x e hpc_y).

# 3.1 Unificando os DataFrames de Locations (apenas anos >= 2017)
df_loc_list = []
for y in range(2017, 2025):
    y_str = str(y)
    if not PROCESSED_DATA[y_str]['Locations'].empty:
        df_loc_list.append(PROCESSED_DATA[y_str]['Locations'])

df_loc_raw = pd.concat(df_loc_list, ignore_index=True) if df_loc_list else pd.DataFrame()

# 3.2 Pivotamento
if not df_loc_raw.empty:
    df_loc_pivot = df_loc_raw.pivot_table(
        index='flare_id',
        columns='coordinate',
        values='flloc_xy',
        aggfunc='first'
    ).reset_index()

    df_loc_pivot = df_loc_pivot.rename(columns={0.0: 'hpc_x', 1.0: 'hpc_y'})
else:
    df_loc_pivot = pd.DataFrame(columns=['flare_id', 'hpc_x', 'hpc_y'])


# 3.3 Filtro do Sub-DataFrame (Explosões sem AR a partir de 2017-02-09)
geom_start_date = pd.to_datetime('2017-02-09', utc=True)

df_sci['matched_AR'] = pd.to_numeric(df_sci['matched_AR'], errors='coerce')
mask_needs_geom = (df_sci['matched_AR'].isna() | (df_sci['matched_AR'] == 0)) & (df_sci['peak_time'] >= geom_start_date)
df_sci_unmatched = df_sci[mask_needs_geom].copy()

# 3.4 Merge (Left Join)
df_sci_geom = pd.merge(
    df_sci_unmatched,
    df_loc_pivot[['flare_id', 'hpc_x', 'hpc_y']],
    on='flare_id',
    how='left'
)

df_sci_geom = df_sci_geom.dropna(subset=['hpc_x', 'hpc_y'])

print(f"✅ Preparações Concluídas:")
print(f"Flares sem AR elegíveis para resgate espacial (>= 2017): {len(df_sci_unmatched)}")
print(f"Flares onde o pivotamento e merge encontraram coordenadas XY com sucesso: {len(df_sci_geom)}")

## Cálculo de Distância Euclidiana (SRS)

In [ ]:
# =============================================================================
# 4.1 FUNÇÕES DE CONVERSÃO DE COORDENADAS (STONYHURST -> HPC)
# =============================================================================
def parse_srs_location(loc_str):
    """Converte string do SRS (ex: 'S24E35') para Latitude e Longitude numéricas."""
    if not isinstance(loc_str, str) or len(loc_str) < 6:
        return np.nan, np.nan
    lat_dir, lat_val = loc_str[0], loc_str[1:3]
    lon_dir, lon_val = loc_str[3], loc_str[4:6]

    try:
        lat = float(lat_val) * (1 if lat_dir == 'N' else -1)
        lon = float(lon_val) * (1 if lon_dir == 'E' else -1)
        return lat, lon
    except ValueError:
        return np.nan, np.nan

def stonyhurst_to_hpc(lat, lon, obstime):
    """Converte coordenadas Stonyhurst Heliográficas para Helioprojective Cartesian (HPC) com base no ponto de vista da Terra no momento da explosão (obstime)."""
    try:
        coord = SkyCoord(lon * u.deg, lat * u.deg,
                         frame="heliographic_stonyhurst",
                         obstime=obstime)
        hpc_coord = coord.transform_to(sunpy.coordinates.Helioprojective(obstime=obstime))

        return hpc_coord.Tx.value, hpc_coord.Ty.value
    except Exception:
        return np.nan, np.nan

In [ ]:
# =============================================================================
# 4.2 & 4.3 CÁLCULO DE DISTÂNCIA EUCLIDIANA (FLARE <-> ARs DO DIA)
# =============================================================================
# O artigo dita a fórmula: Distance = sqrt((X_AR - X_Flare)^2 + (Y_AR - Y_Flare)^2)
df_sci_geom['nearest_AR_srs'] = np.nan
df_sci_geom['min_distance_arcsec'] = np.nan

print("Iniciando Cálculo de Distâncias Geométricas (Isso pode ser demorado devido à conversão do SunPy)...")

for idx, flare in tqdm(df_sci_geom.iterrows(), total=len(df_sci_geom), desc="Calculando HPC e Distâncias"):
    flare_time = flare['peak_time']
    flare_date = flare_time.date()

    flare_hpc_x = flare['hpc_x']
    flare_hpc_y = flare['hpc_y']

    active_ars_today = df_srs_raw[df_srs_raw['Date'] == flare_date]

    if active_ars_today.empty:
        continue

    min_dist = np.inf
    best_ar = np.nan

    # Itera sobre as ARs do dia para encontrar a mais próxima
    for _, ar_row in active_ars_today.iterrows():
        lat, lon = parse_srs_location(ar_row['Location'])

        if pd.isna(lat) or pd.isna(lon):
            continue

        # Converte a posição da AR para HPC no momento exato do pico da explosão
        ar_hpc_x, ar_hpc_y = stonyhurst_to_hpc(lat, lon, flare_time)

        if pd.isna(ar_hpc_x) or pd.isna(ar_hpc_y):
            continue

        distance = np.sqrt((ar_hpc_x - flare_hpc_x)**2 + (ar_hpc_y - flare_hpc_y)**2)

        if distance < min_dist:
            min_dist = distance
            best_ar = ar_row['Nmbr']

    if min_dist != np.inf:
        df_sci_geom.at[idx, 'nearest_AR_srs'] = best_ar
        df_sci_geom.at[idx, 'min_distance_arcsec'] = min_dist

print("\n Cálculo Geométrico Concluído!")

## Atribuição Final por Geometria

In [ ]:
# Para garantir a confiabilidade, o artigo determina que apenas atribuições com distância estritamente menor que 250 arcsec sejam retidas.
# 5.1 Aplicar o limiar (Threshold)
threshold_arcsec = 250.0
df_sci_geom['rescued_AR'] = np.where(
    df_sci_geom['min_distance_arcsec'] < threshold_arcsec,
    df_sci_geom['nearest_AR_srs'],
    np.nan
)

rescued_count = df_sci_geom['rescued_AR'].notna().sum()
print(f"Flares resgatados com sucesso via geometria (distância < {threshold_arcsec} arcsec): {rescued_count}")

In [ ]:
# 5.2 Atualizar a base principal (df_sci) com os dados resgatados
map_rescued_ar = df_sci_geom.dropna(subset=['rescued_AR']).set_index('flare_id')['rescued_AR']
mask_to_update = df_sci['flare_id'].isin(map_rescued_ar.index)

df_sci.loc[mask_to_update, 'matched_AR'] = df_sci.loc[mask_to_update, 'flare_id'].map(map_rescued_ar)
df_sci.loc[mask_to_update, 'match_source'] = 'Geometry_SRS'

In [ ]:
# 5.3 Validação de Sanidade
total_sci = len(df_sci)
final_matched = df_sci['matched_AR'].notna().sum()

print("\n--- Resumo Final de Atribuição de ARs ---")
print(f"Total de flares Science-Quality: {total_sci}")
print(f"Total com AR atribuída: {final_matched} ({(final_matched / total_sci * 100):.1f}%)")
print(f"Total mantidos como 'Sem AR': {total_sci - final_matched}")

## Consolidação e Exportação (Trusted/Clean Layer)

In [ ]:
# 6.1 Padronização de Nulos e Tipagem
df_sci['match_source'] = df_sci['match_source'].fillna('UNMATCHED')
df_sci['matched_AR'] = pd.to_numeric(df_sci['matched_AR'], errors='coerce').astype('Int64')

In [ ]:
# 6.2 Organização Estrutural
df_final = df_sci.sort_values(by='peak_time').reset_index(drop=True)
df_final = df_final.rename(columns={
    'matched_AR': 'active_region_no',
    'match_source': 'ar_provenance'
})

In [ ]:
# 6.3 Exportação
# Isolar os dados finais em uma camada limpa (Trusted)
trusted_dir = os.path.join(PROCESSED_PATH, 'Trusted')
os.makedirs(trusted_dir, exist_ok=True)

output_filepath = os.path.join(trusted_dir, 'Augmented_Science_Quality_Flare_List.csv')
df_final.to_csv(output_filepath, index=False)

print("✅ Pipeline de Tratamento Finalizado com Sucesso!")
print(f"Dataset exportado para: {output_filepath}")
print(f"Dimensões finais da matriz: {df_final.shape}")
print("\nAmostra dos dados finais:")
display(df_final.head())